In [0]:
from pyspark.sql import functions as F

oncology_pat_ids = (
    spark.table("datahub_dev_bronze.datahub_clarity.mv_dm_patient_access")
    .alias("pa")
    .join(
        spark.table("opsanalytics_adb_workspace01.oncology.oncology_department_groupings").alias("dg"),
        on=F.col("pa.DEPARTMENT_ID") == F.col("dg.DEPARTMENT_ID"),
        how="inner",
    )
    .select(F.col("pa.PAT_ID").alias("PAT_ID"))
    .distinct()
)

In [0]:
oracle_jdbc_url = dbutils.secrets.get(scope="oao_secrets", key="ORACLE_JDBC_URL")
oracle_user = "OAO_PRODUCTION"
oracle_password = dbutils.secrets.get(scope="oao_secrets", key="OAO_PRODUCTION")

demographics_df = (
    spark.read
    .format("jdbc")
    .option("url", oracle_jdbc_url)
    .option("dbtable", "MV_PATIENT_SELECT_DEMOGRAPHICS")
    .option("user", oracle_user)
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("oracle.net.ssl_server_dn_match", "true")
    .load()
)

filtered_demographics_df = demographics_df.join(
    oncology_pat_ids,
    on="PAT_ID",
    how="left_semi"
)

In [0]:
filtered_demographics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("opsanalytics_adb_workspace01.oncology.MV_PATIENT_SELECT_DEMOGRAPHICS")